# GPU OOM note
If you hit **CUDA out of memory** while loading the model, it's often because VRAM is still held by the current Python process (even after `del` + `empty_cache`).
- Prefer restarting the notebook kernel to fully release VRAM.
- If you’re on a ~16GB GPU, consider using a smaller model (e.g. `Qwen/Qwen2.5-3B-Instruct`) or let Transformers **CPU-offload** some layers via `device_map="auto"`.
This notebook cell below is configured to be more VRAM-friendly by default.

In [ ]:
import os
import gc
import re
import string

# Must be set before importing torch for best effect.
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")
os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")

import torch
from opencc import OpenCC
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig


def _gpu_free_gb() -> float:
    if not torch.cuda.is_available():
        return 0.0
    free, _total = torch.cuda.mem_get_info()
    return free / 2**30


def _print_gpu_mem(prefix: str = "") -> None:
    if not torch.cuda.is_available():
        print(prefix + "CUDA not available")
        return
    free, total = torch.cuda.mem_get_info()
    print(prefix + f"CUDA free: {free / 2**30:.2f} GiB / total: {total / 2**30:.2f} GiB")


# Default model choice
# - Set `HF_MODEL_ID` to override (e.g. Qwen/Qwen2.5-7B-Instruct)
# - This notebook defaults to 3B for reliability on ~16GB GPUs.
MODEL_ID = os.getenv("HF_MODEL_ID", "Qwen/Qwen2.5-3B-Instruct")

# If you re-run this cell, aggressively free VRAM first.
globals().pop("model", None)
globals().pop("tokenizer", None)

gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
    torch.cuda.ipc_collect()

_print_gpu_mem(prefix="Before load: ")

# If VRAM is basically full, even 4-bit loading will fail. In that case, we fall back to CPU.
# (Best fix is still: restart the kernel / close other GPU apps.)
use_cpu_fallback = torch.cuda.is_available() and _gpu_free_gb() < 2.0
if use_cpu_fallback:
    print("WARNING: Very low free VRAM detected. Falling back to CPU-only model load.")
    print("To use GPU: restart the notebook kernel and close other GPU processes.")
    MODEL_ID = os.getenv("HF_MODEL_ID_CPU", "Qwen/Qwen2.5-1.5B-Instruct")

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, use_fast=True)
if tokenizer.pad_token_id is None:
    tokenizer.pad_token = tokenizer.eos_token

if use_cpu_fallback:
    # CPU path: do not use bitsandbytes quantization (CUDA-only).
    model = AutoModelForCausalLM.from_pretrained(
        MODEL_ID,
        device_map="cpu",
        torch_dtype=torch.float32,
        low_cpu_mem_usage=True,
    )
else:
    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_use_double_quant=True,
        bnb_4bit_compute_dtype=torch.float16,
    )

    # Use auto device mapping so Transformers can offload to CPU if needed.
    device_map = "auto"
    total_gb = int(torch.cuda.get_device_properties(0).total_memory / 2**30)
    gpu_budget = max(1, total_gb - 2)
    max_memory = {0: f"{gpu_budget}GiB", "cpu": "64GiB"}

    model = AutoModelForCausalLM.from_pretrained(
        MODEL_ID,
        quantization_config=bnb_config,
        device_map=device_map,
        max_memory=max_memory,
        torch_dtype=torch.float16,
        low_cpu_mem_usage=True,
    )

model.eval()

_opencc_s2t = OpenCC("s2t")

_CJK_PUNCT = set("，。！？；：、（）《》「」『』【】—…")
_ASCII_PUNCT = set(string.punctuation)
_PUNCT = _CJK_PUNCT | _ASCII_PUNCT | {" ", "\n", "\t"}

# Allow: CJK Unified + Ext A, common punctuation, and whitespace.
# This excludes latin letters/digits and most symbols.
_ALLOWED_RE = re.compile(r"^[\u3400-\u4dbf\u4e00-\u9fff，。！？；：、（）《》「」『』【】—…\s]+$")
_TERMINAL_PUNCT = ("。", "！", "？")


def _count_non_punct_chars(text: str) -> int:
    return sum(1 for ch in text if ch not in _PUNCT)


def _contains_exactly_once(text: str, word: str) -> bool:
    return text.count(word) == 1


def _has_long_repeats(text: str, max_run: int = 3) -> bool:
    run = 1
    for a, b in zip(text, text[1:]):
        if a == b and a not in _PUNCT:
            run += 1
            if run > max_run:
                return True
        else:
            run = 1
    return False


def _looks_like_stutter(text: str, word: str) -> bool:
    # Heuristic for things like “可可怕” when target word is “可怕”.
    if len(word) == 2:
        return (word[0] + word) in text
    return False


def _sanitize_one_line(text: str) -> str:
    # Take first non-empty line, strip quotes/spaces.
    lines = [ln.strip() for ln in text.splitlines() if ln.strip()]
    text = lines[0] if lines else text.strip()
    text = text.strip('"“”')
    return text


def _passes_language_filters(text: str) -> bool:
    # Ban latin letters / digits / weird symbols by only allowing a strict charset.
    if not _ALLOWED_RE.match(text):
        return False
    # Avoid stray ASCII punctuation like ?>
    if any(ch in _ASCII_PUNCT for ch in text):
        return False
    return True


def _validate_candidate(
    text: str,
    word: str,
    *,
    min_chars: int,
    max_chars: int,
    require_terminal_punct: bool,
    avoid_stutter: bool,
    max_repeat_run: int,
 ):
    """Return (ok, reason)."""
    if not text:
        return False, "empty"

    if not _passes_language_filters(text):
        return False, "non_chinese_or_ascii"

    if require_terminal_punct and not text.endswith(_TERMINAL_PUNCT):
        return False, "missing_terminal_punct"

    if not _contains_exactly_once(text, word):
        return False, "word_not_exactly_once"

    char_count = _count_non_punct_chars(text)
    if char_count < min_chars:
        return False, "too_short"
    if char_count > max_chars:
        return False, "too_long"

    if _has_long_repeats(text, max_run=max_repeat_run):
        return False, "repetition"

    if avoid_stutter and _looks_like_stutter(text, word):
        return False, "stutter"

    return True, "ok"


def generate_sentences(
    word: str,
    meaning: str = "",
    level: str | int | None = None,
    num_sentences: int = 3,
    min_chars: int | None = None,
    max_chars: int | None = None,
    script: str = "traditional",  # "traditional" | "simplified" | "auto"
    max_retries_per_sentence: int = 20,
    mode: str = "balanced",  # "strict" | "balanced"
    avoid_stutter: bool = True,
    failure_marker: str = "[FAILED]",
 ):
    """Generate example sentences.

    `mode="balanced"` is recommended for bulk generation (fewer failures).
    `level` (1..6) is used to steer difficulty (soft constraint).

    Defaults:
    - Traditional Chinese output (`script="traditional"`)
    - Balanced validation: wider length, terminal punctuation optional
    """

    # Defaults tuned for fewer failures
    if mode not in {"strict", "balanced"}:
        raise ValueError("mode must be 'strict' or 'balanced'")

    if min_chars is None:
        min_chars = 8 if mode == "balanced" else 10
    if max_chars is None:
        max_chars = 26 if mode == "balanced" else 20

    require_terminal_punct = (mode == "strict")
    max_repeat_run = 3 if mode == "balanced" else 2

    system = (
        "You generate natural Mandarin example sentences for Chinese learners. "
        "Follow constraints exactly. Output only the sentence text."
    )

    if script == "traditional":
        script_line = "Use Traditional Chinese characters ONLY. Do not use Simplified."
    elif script == "simplified":
        script_line = "Use Simplified Chinese characters ONLY."
    else:
        script_line = "Use natural Chinese characters matching the input word."

    if level is None:
        level_lines = ""
    else:
        level_lines = (
            f"- Target difficulty: TOCFL Level {level}\n"
            f"- Prefer vocabulary and grammar appropriate for that level\n"
            f"- Avoid proper nouns and avoid rare idioms\n"
        )

    results: list[str] = []

    for _ in range(num_sentences):
        accepted: str | None = None
        last_text = ""
        last_reason = ""

        for attempt in range(max_retries_per_sentence):
            # Add feedback after the first failed attempt to improve compliance.
            feedback = ""
            if attempt > 0 and last_text:
                feedback = (
                    "Previous attempt (invalid): "
                    f"{last_text}\n"
                    f"Reason: {last_reason}\n"
                    "Rewrite to satisfy ALL constraints.\n"
                )

            user = (
                f"Target word: {word}\n"
                f"Meaning (English): {meaning}\n"
                + (f"TOCFL level: {level}\n" if level is not None else "")
                + "Constraints:\n"
                "- Output ONE sentence only\n"
                "- Do not include quotes, bullet points, numbering, or explanations\n"
                "- Do not include English words, names, roman letters, or digits\n"
                "- No emojis\n"
                f"- Must include the target word exactly once\n"
                f"- Length: {min_chars}–{max_chars} Chinese characters (excluding punctuation)\n"
                "- Everyday context\n"
                f"- {script_line}\n"
                + (level_lines if level_lines else "")
                + ("- End the sentence with 。 or ！ or ？\n" if require_terminal_punct else "")
                + ("\n" + feedback if feedback else "")
            )

            messages = [
                {"role": "system", "content": system},
                {"role": "user", "content": user},
            ]

            prompt = tokenizer.apply_chat_template(
                messages, tokenize=False, add_generation_prompt=True
            )
            inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

            with torch.no_grad():
                out = model.generate(
                    **inputs,
                    max_new_tokens=64,
                    do_sample=True,
                    temperature=0.65 if mode == "balanced" else 0.7,
                    top_p=0.9,
                    repetition_penalty=1.18,
                    no_repeat_ngram_size=3,
                    pad_token_id=tokenizer.eos_token_id,
                )

            new_tokens = out[0, inputs["input_ids"].shape[-1] :]
            text = tokenizer.decode(new_tokens, skip_special_tokens=True)
            text = _sanitize_one_line(text)

            # Enforce script conversion
            if script == "traditional":
                text = _opencc_s2t.convert(text)

            ok, reason = _validate_candidate(
                text,
                word,
                min_chars=min_chars,
                max_chars=max_chars,
                require_terminal_punct=require_terminal_punct,
                avoid_stutter=avoid_stutter,
                max_repeat_run=max_repeat_run,
            )

            if ok:
                accepted = text
                break

            last_text = text
            last_reason = reason

        results.append(accepted if accepted is not None else failure_marker)

    return results


In [ ]:
prompt = "可怕"
# Balanced mode => fewer [FAILED] while still filtering English/garbage.
generated_sentences = generate_sentences(
    prompt,
    meaning="scary/terrifying",
    level=1,
    num_sentences=3,
    mode="balanced",
)
for i, sentence in enumerate(generated_sentences, 1):
    print(f"Sentence {i}:\n{sentence}\n")


In [ ]:
sentences_per_word_map = {}
processed_words = set()

In [ ]:
# Generate Traditional Chinese sentences for the top words in the dataset.
import pandas as pd
df = pd.read_csv("chinese_8000_tocfl_translated_merged.csv")

level_mapping = {
    "入門級(Level 1)": 1,
    "準備級一級(Novice 1)": 1,
    "準備級二級(Novice 2)": 1,
    "基礎級(Level 2)": 2,
    "進階級(Level 3)": 3,
    "高階級(Level 4)": 4,
    "流利級(Level 5)": 5,
    "6": 6,
}

index2 = 0
start_index = 5000  # Change this to resume from a specific index.

for index, row in df.head(6000).iterrows():
    if index < start_index:
        continue
    # print(row)
    print(f"Indexé = {index2}")
    word = row["Vocabulary"]

    # When there is a / in the word, take the left word
    if '/' in word:
        word = word.split('/')[0]
    print(f"Processed word = {word}")

    if '(' in word:
        word = word.split('(')[0].strip()
    

    if word in processed_words:
        continue

    meaning = row["Translation"]

    raw_level = row["Level"]
    tocfl_level = level_mapping.get(raw_level, 1)

    print(f"Final word = {word} with level : {raw_level} (mapped to {tocfl_level}) ")

    print(f"Generating sentences for: {word} ({meaning}), TOCFL Level {tocfl_level}")

    generated_sentences = generate_sentences(
        word,
        meaning=meaning,
        level=tocfl_level,
        num_sentences=2,
        script="traditional",
        mode="balanced"
    )

    sentences_per_word_map[word] = generated_sentences
    processed_words.add(word)
    index2 += 1


    # Limit to 5 words for demo purposes.
    if index2 > 200 :
        break


    for i, sentence in enumerate(generated_sentences, 1):
        print(f"Sentence {i}:\n{sentence}\n")

# Save generated sentences to a CSV file.
import csv
with open("generated_sample_sentences_traditional.csv", "w", newline="", encoding="utf-8") as csvfile:
    writer = csv.writer(csvfile)
    # Header : word  sentence
    # One sentence per row
    writer.writerow(["word", "sentence"])
    for word, sentences in sentences_per_word_map.items():
        for sentence in sentences:
            writer.writerow([word, sentence])